# 1. Agent高级用法-结构化输出：用户只需需要通过“response-format”参数设置期望输出格式，agent通过“structured_response"返回

## 结构化输出4种策略

### 1.1 ProviderStrategy策略 （它只支持原生模型如openai、等等，对DeepSeek模型不支持，下面运行错误）

In [3]:
from langchain.agents.structured_output import ProviderStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

class Person(BaseModel):
    """用户信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话")


model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
)
# 创建agent
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(Person)
)


messages = {
    "messages":[
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
}

res = agent.invoke(messages)
rprint(res)

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

### 1.2 ToolStrategy策略，基本上都支持，相当于一个虚拟工具，只要模型支持调用工具，此方式就可行

In [5]:
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

class Person(BaseModel):
    """用户信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话")


model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    # deepseek模型需要关闭推理模式，否则也不行，会出现冲突；Thinking 模式不支持强制 tool_choice
    extra_body={"thinking":{"type":"disabled"}}
)
# 创建agent
agent = create_agent(
    model=model,
    response_format=ToolStrategy(Person)
)


messages = {
    "messages":[
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
}

res = agent.invoke(messages)
rprint(res)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='00404311-808c-4898-b70f-a111f51b893d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 346,
                    'total_tokens': 423,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    },
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 346
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '5b652ecf-9c04-4cd3-a949-ed6d9d7e3760',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffa50-2f25-7c20-b106-7dff5074e9ac-0',
            tool_calls=[
                {
                    'name': 'Person',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_GyCAX10V4MawDEew9gQy4451',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 77,
                'total_tokens': 423,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='Person',
            id='eca963f3-9f4d-4408-924e-fea059a89320',
            tool_call_id='call_00_GyCAX10V4MawDEew9gQy4451'
        )
    ],
    'structured_response': Person(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

### 1.3 type/AutoStrategy策略

In [6]:
from langchain.agents.structured_output import AutoStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

class Person(BaseModel):
    """用户信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话")


model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    # deepseek模型需要关闭推理模式，否则也不行，会出现冲突；Thinking 模式不支持强制 tool_choice
    extra_body={"thinking":{"type":"disabled"}}
)
# 创建agent
agent = create_agent(
    model=model,
    response_format=AutoStrategy(Person)
)


messages = {
    "messages":[
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
}

res = agent.invoke(messages)
rprint(res)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='6d1ba292-7da0-4976-acb6-28906b8fe1ff'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 346,
                    'total_tokens': 423,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 90
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'b3f026cf-5315-4bfe-acb1-0f1f75a6b140',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffa53-0035-7f41-a48c-643cff757677-0',
            tool_calls=[
                {
                    'name': 'Person',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_eAWzDGG80YBS8Be30j6A4383',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 77,
                'total_tokens': 423,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='Person',
            id='1d2b5f77-8a15-4331-9578-fdb55031b93c',
            tool_call_id='call_00_eAWzDGG80YBS8Be30j6A4383'
        )
    ],
    'structured_response': Person(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

### 1.4 None 默认策略，不以结构化输出